In [ ]:
# Run from the repo root so relative imports and `!`-shell commands resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


In [ ]:
from eval.verification import load_bin
import torch

In [ ]:
# TODO: local paths in this cell were redacted to ""; set them before running.
lfw = load_bin('', (112, 112))
cfp = load_bin('', (112, 112))
age = load_bin('', (112, 112))

In [ ]:
@torch.no_grad()
def feat_ext(dataset, backbone, batch_size, device):
    data, issame = dataset[0], dataset[1]
    len_data = data[0].size(0)
    feats = []
    for dset in data:
        tmp = []
        ba = 0
        while ba < len_data:
            bb = min(ba + batch_size, len_data)
            img = dset[ba:bb]
            img = (img - 127.5) / 127.5
            feat = F.normalize(backbone(img.to(device)))
            tmp.append(feat)
            ba = bb
        tmp = torch.cat(tmp, dim = 0)
        feats.append(tmp)
        
    feats = F.normalize(feats[0] + feats[1])
    return feats

In [ ]:
@torch.no_grad()
def eval_perf(feats, tau_upper = 1, target_far = None, batch_size = 512):
    device = feats.device

    left, right = feats[::2], feats[1::2]
    len_pairs = left.size(0)
    stepsize = len_pairs // 20
    issame = []
    
    for i in range(len_pairs):
        if (i//stepsize) % 2 == 0:
            issame += [True]
        else:
            issame += [False]
    
    issame = torch.BoolTensor(issame).to(device)
    
    best_acc = 0
    best_tau = 0
    best_tar = 0
    best_far = 0
    
    acc = []
    thx = []
    tar = []
    far = []
    
    score = F.cosine_similarity(left, right)
    taus = torch.linspace(-1, 1, 2000)
    
    for tau in taus:
        pred = score > tau
    
        # Evaluation
        TA = (pred & issame).sum()
        TR = (~pred & ~issame).sum()
        FA = (pred & ~issame).sum()
        FR = (~pred & issame).sum()

        # Eval Criteria
        TAR = TA / (len_pairs / 2)
        FAR = FA / (len_pairs / 2)
        ACC = (TA + TR) / len_pairs
        
        if ACC > best_acc:
            best_acc = ACC
            best_tar = TAR
            best_far = FAR
            best_tau = tau
        
        if (far != None):
            acc.append(ACC)
            tar.append(TAR)
            far.append(FAR)
            thx.append(tau)
            
    if (target_far != None):
        far = torch.Tensor(far)
        diff = far - target_far
        ind = diff.abs().argmin()
        return tar[ind], far[ind], acc[ind], thx[ind]    
    else:        
        return best_tar, best_far, best_acc, best_tau

In [ ]:
from backbones.iresnet import *
from backbones.mobilefacenet import *
from backbones import get_model
import torch.nn.functional as F

In [ ]:
def model_bench3set(backbone):
    lfw_feat = feat_ext(lfw, backbone, 512, device)
    cfp_feat = feat_ext(cfp, backbone, 512, device)
    age_feat = feat_ext(age, backbone, 512, device)
    
    print("LFW Eval..")
    best_tar, best_far, best_acc, best_tau = eval_perf(lfw_feat)
    print('TAR@FAR: {:.4f}@{:.4f}'.format(best_tar, best_far))
    print('ACC: {:.2f}%'.format(best_acc*100))
    print('Tau: {:.4f}'.format(best_tau))
    print()
    
    print("CFP-FP Eval..")
    best_tar, best_far, best_acc, best_tau = eval_perf(cfp_feat)
    print('TAR@FAR: {:.4f}@{:.4f}'.format(best_tar, best_far))
    print('ACC: {:.2f}%'.format(best_acc*100))
    print('Tau: {:.4f}'.format(best_tau))
    print()
    
    print("AGE-DB Eval..")
    best_tar, best_far, best_acc, best_tau = eval_perf(age_feat)
    print('TAR@FAR: {:.4f}@{:.4f}'.format(best_tar, best_far))
    print('ACC: {:.2f}%'.format(best_acc*100))
    print('Tau: {:.4f}'.format(best_tau))

In [ ]:
device = 'cuda:1'

## Benchmark any checkpoint

`model_bench3set(backbone)` runs LFW / CFP-FP / AgeDB on a single eval-mode backbone.
To benchmark your own checkpoints:

1. Set `device` and the checkpoint path.
2. Build the matching backbone via `get_model(arch)` or one of `iresnet18/50/100`,
   then `load_state_dict`. For ONNX files, use `onnx2torch.convert(path)` instead.
3. Move to `device` and call `model_bench3set(backbone)`.

Repeat for each checkpoint you want to evaluate.


In [ ]:
# Example: state_dict checkpoint (PyTorch .pt / .pth)
# TODO: local paths in this cell were redacted to ""; set them before running.
device    = 'cuda:0'
model_dir = ''          # path to checkpoint
arch      = 'r50'       # one of {'mbf', 'r18', 'r50', 'r100', 'vit_b'}

backbone = get_model(arch, fp16=False)
backbone.load_state_dict(torch.load(model_dir, weights_only=True))
backbone = backbone.eval().to(device)

model_bench3set(backbone)


In [ ]:
# Example: ONNX checkpoint
# TODO: local paths in this cell were redacted to ""; set them before running.
device    = 'cuda:0'
onnx_path = ''          # path to .onnx file

model = convert(onnx_path).eval().to(device)
model_bench3set(model)
